# Publish a whole batch (no dimension/coord reduction)

Converts every field of every `.mat` under one **tag** at native resolution to one Zarr store per run.

Each store keeps the run's axes:

| dim | size | what |
|---|---|---|
| `time` | `100·tf + 1` | fd15's grid, years since first application |
| `month` | `12·tf + 1` | the cumulative mass series |
| `driver_month` | `12·tf` | the monthly forcing, and `pH_monthly` |
| `depth` | 31 | 0–0.15 m |
| `mineral`, `alk_row`, `element` | 9, 9, 14 | named coordinates |
| `point`, `soil_level` | varies, 7 | the cluster's deployed points; the soil profile |



In [ ]:
import re, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import fsspec, numpy as np, pandas as pd, scipy.io as sio, s3fs, xarray as xr
from beerling_analysis import io

warnings.filterwarnings("ignore")

TAG        = "e0-conus-1gt"
ROOT       = "s3://carbonplan-carbon-removal/ew-workflows-data/beerling-2025-model-extension"
OUT        = f"{ROOT}/postproc/{TAG}/all"   # set to a local path to write locally instead
CACHE      = Path(io.CACHE) / TAG
KEEP_CACHE = False     # False -> delete each .mat once converted
SKIP_DONE  = True      # True  -> resume, leaving stores that already exist
WORKERS    = 8

fs  = s3fs.S3FileSystem()             # the results bucket the .mat come from
ofs = fsspec.core.url_to_fs(OUT)[0]   # wherever OUT points -- S3 or local
CACHE.mkdir(parents=True, exist_ok=True)
print(f"{TAG}  ->  {OUT}")

## 1. The runs

Use the filename as the manifest: `{experiment}_drv{drv}_{state}_{mineralogy}_{tf}yr.mat`

In [ ]:
PAT = re.compile(r"^(?P<exp>E\d)_drv(?P<drv>\d+)_(?P<source>.+)_(?P<feedstock>[12])_(?P<tf>\d+)yr\.mat$")

runs = pd.DataFrame([
    {**PAT.match(n).groupdict(), "file": n}
    for n in (Path(k).name for k in fs.ls(f"{ROOT}/results/{TAG}"))
    if PAT.match(n)
]).astype({"drv": int, "feedstock": int, "tf": int}).sort_values(["drv", "feedstock"])
runs["start_year"] = 2071 - runs.tf
runs = runs.reset_index(drop=True)
print(f"{len(runs)} runs, {runs.drv.nunique()} clusters, {runs.tf.sum():,} simulated years")

## 2. Convert `.mat` to `Dataset`


In [ ]:
ELEMS  = ("Ca", "Mg", "K", "Na", "Si", "Fe", "P")
GROUPS = (("total", None), ("basalt", "Basalt"), ("solution", "Solution"), ("outflow", "Outflow"))

A  = lambda x: np.asarray(x, float)                        # array, as-is
F  = lambda x: float(np.atleast_1d(x).ravel()[0])          # scalar
S  = lambda x: str(np.atleast_1d(x).ravel()[0])            # string
SL = lambda x: [str(e) for e in np.atleast_1d(x).ravel()]  # list of strings

def to_dataset(m):
    """Every field of one run, nothing reduced."""
    d, T, I = m["totals"].diag, m["totals"], m["inps"]
    t, y0 = A(d.t), int(F(m["start_year"]))
    tf, en = int(round(float(t[-1]))), list(d.elements._fieldnames)

    v = {k: (("depth", "time"), A(getattr(d, k)))
         for k in ("pH", "Ca_sol", "Ca_sat", "Ca_precip", "H_sol")}
    v |= {k: (("time",), A(getattr(d, k))) for k in ("pH_avg", "q")}
    v |= {k: (("mineral", "depth", "time"), A(getattr(d, k))) for k in ("B", "C")}
    v |= {k: (("mineral",), A(getattr(d, k))) for k in ("mol_mass", "mf")}
    v |= {k: (("month",), A(m[k])) for k in
          ("CO2_mass", "CO2_mass_outflow", "CO2_mass_precip", "c2b_ratio",
           "Application_Basalt_mass", "K_mass", "P_mass")}
    v |= {f"driver_{k}": (("driver_month",), A(getattr(I, k)))
          for k in ("co2", "npp", "sr", "wat")}
    for pre, g in GROUPS:                      # Si/Fe/P are guarded upstream
        g = T if g is None else getattr(T, g)
        v |= {f"{pre}_{e}": (("time",), A(getattr(g, e))) for e in ELEMS if hasattr(g, e)}
    v |= dict(
        alk=(("alk_row", "depth", "time"), A(d.alk)),
        mineral_balance=(("mineral", "time"), A(T.tot)),
        soil_CO2_static=(("depth",), A(d.soil_CO2)),
        stoichiometry=(("element", "mineral"), np.stack([A(getattr(d.elements, e)) for e in en])),
        pH_monthly=(("depth", "driver_month"), A(m["pH"])),
        driver_st=(("driver_month", "soil_level"), A(I.st)),
        driver_alk=(("driver_month", "depth"), A(I.alk)),
        pH0=(("soil_level",), np.atleast_1d(A(I.pH))),
        point_ind=(("point", "pair"), np.atleast_2d(A(I.ind))))

    return xr.Dataset(v, coords=dict(
        time=t, depth=A(d.z), mineral=SL(d.mineral_names), alk_row=SL(d.alk_rows),
        element=en, month=np.arange(12 * tf + 1) / 12,
        driver_month=np.arange(1, 12 * tf + 1) / 12,
        soil_level=np.arange(1, np.atleast_1d(A(I.pH)).size + 1),
        point=np.arange(np.atleast_1d(A(I.ar)).size),
        calendar_year=("time", y0 + t - 1),
        area_ha=("point", np.atleast_1d(A(I.ar))),
        lind=("point", np.atleast_1d(A(I.lind)).astype("int64")),
        recipient=("point", SL(I.nam_l)), state_code=("point", SL(I.nam_s))),
    attrs=dict(
        drv=int(F(m["drv"])), start_year=y0, tf=tf, experiment=S(m["experiment"]),
        basalt=S(m["basalt"]), buffer=S(d.buffer), crop=S(I.nam),
        wall_seconds=F(m["wall_seconds"]), phi=F(d.phi), soil_depth_m=F(d.depth),
        b_mass=F(d.b_mass), p80_um=F(I.p80),
        **{k: F(getattr(I, k)) for k in ("fc", "lamb", "lambi", "spin", "scen", "yy")},
        grid="time = years since first application on fd15's 100/yr grid; month = the "
             "12/yr cumulative-mass grid; driver_month = the 12/yr forcing grid",
        units="B, C kg/m3 (C/mol_mass = mol/L); pH*, Ca_sol, Ca_sat, Ca_precip, alk, "
              "H_sol per depth node (Ca_* and alk mol/L, H_sol nitrification acidity); "
              "q m/yr effective post-bucket flux; total_/basalt_/solution_/outflow_ "
              "mol/m2; CO2_mass* and Application_Basalt_mass g/m2; K_mass, P_mass kg/m2; "
              "mineral_balance = B_mass + C_mass*phi + outflow (mass_balance.m:39)",
        dropped="totals.Precip.Ca (bit-identical to Ca_precip); top-level Ca/Mg/K/Na and "
                "Outflow.*_monthly_mols (interp1 of outflow_* onto 0:1/12:tf, "
                "mass_balance.m:203,317); inps.ibas and inps.iprof (both equal the "
                "basalt attr); oor (empty struct). Everything else is here.",
        tag=TAG, source=f"{ROOT}/results/{TAG}/"))

def load(name):
    """Stage one .mat from S3 if it is not already cached, and read it."""
    p = CACHE / name
    if not p.exists():
        fs.get(f"{ROOT}/results/{TAG}/{name}", str(p))
    return sio.loadmat(p, squeeze_me=True, struct_as_record=False)

ds0 = to_dataset(load(runs.file.iloc[0]))
print(f"{runs.file.iloc[0]}\n  {dict(ds0.sizes)}\n  {len(ds0.data_vars)} vars, "
      f"{sum(x.nbytes for x in ds0.data_vars.values()) / 1e6:.0f} MB")
ds0

## 3. Convert

One zarr store per run that is written from the `.mat` file.

In [ ]:
done = lambda p: any(ofs.exists(f"{p}/{f}") for f in ("zarr.json", ".zmetadata"))

def convert(r):
    dest = f"{OUT}/{Path(r.file).stem}.zarr"
    if SKIP_DONE and done(dest):
        return dest, np.nan
    try:
        ds = to_dataset(load(r.file))
    finally:
        if not KEEP_CACHE:
            (CACHE / r.file).unlink(missing_ok=True)
    ds.to_zarr(dest, mode="w", consolidated=True)
    return dest, sum(x.nbytes for x in ds.data_vars.values()) / 1e6

with ThreadPoolExecutor(WORKERS) as ex:
    out = []
    for i, res in enumerate(ex.map(convert, runs.itertuples()), 1):
        out.append(res)
        print(f"  {i}/{len(runs)}", end="\r")

runs["store"], runs["MB"] = zip(*out)
runs.to_csv(f"{OUT}/manifest.csv", index=False)
done = runs.MB.notna().sum()
print(f"{done} converted, {len(runs) - done} already present, "
      f"{runs.MB.sum() / 1e3:.1f} GB written -> {OUT}/manifest.csv")

## 4. Read one back

Check that the saved results are intact.

In [ ]:
man = pd.read_csv(f"{OUT}/manifest.csv")
ds = xr.open_zarr(man.store.iloc[0])
print(f"{man.file.iloc[0]}  drv{ds.attrs['drv']} {ds.attrs['basalt']} "
      f"{ds.attrs['tf']} yr from {ds.attrs['start_year']}, buffer={ds.attrs['buffer']}")
print(f"  bytownite undissolved: {float(ds.B.sel(mineral='Bytownite').isel(time=0).sum()):.1f} "
      f"-> {float(ds.B.sel(mineral='Bytownite').isel(time=-1).sum()):.2f} kg/m3")
print(f"  Ca budget at 2070 (mol/m2): applied {float(ds.total_Ca[-1]):.1f} = "
      f"undissolved {float(ds.basalt_Ca[-1]):.1f} + dissolved {float(ds.solution_Ca[-1]):.3f} "
      f"+ exported {float(ds.outflow_Ca[-1]):.1f} + precipitated "
      f"{float(ds.total_Ca[-1] - ds.basalt_Ca[-1] - ds.solution_Ca[-1] - ds.outflow_Ca[-1]):.1f}")
ds

In [ ]:
# ---